# Preprocessing glacier datasets

In [ ]:
import os 
import glob
import numpy as np
import pandas as pd
import geopandas as gpd

# Import libraries
import rioxarray as rio
from rioxarray.merge import merge_arrays
from rasterio.enums import Resampling

from andeangc.config import find_repo_root, get_dir, get

wd = find_repo_root()
os.chdir(wd)

# the glacier data is spread over several directories under data_root; address
# them explicitly so the notebook works from any working directory
path_dhdt      = get_dir('glacier_dhdt')
path_thickness = get_dir('glacier_thickness')
path_outlines  = get_dir('glacier_outlines')
inputs         = get('inputs')
outputs        = get('outputs')
nodata         = get('nodata_value')

#TODO CHECK FINAL RES AND RESAMPLING METHOD (conservative?)
#TODO optimize code (functions?)

## 1. **Hugonnet et al. 2021**
- Merges multiple glacier elevation change rate (dhdt) raster files
- Reprojects data to EPSG:32718 (UTM Zone 18S) and then to EPSG:4326 (WGS84)
- Resamples to 100m resolution
- Converts ice thickness to water equivalent (mm) using density factor (1.091)
- **Output**: `dhdt_2000_2020_hugonnet.tif` (glacier mass balance 2000-2020)


In [ ]:
## dhdt
res_hugonnet = get('res_hugonnet')
list_files = glob.glob(str(path_dhdt / inputs['glacier_dhdt'] / "*.tif"))

glacier_list = []

# Read rasters file
for glacier in list_files:
    glacier_i = rio.open_rasterio(glacier, chunks = "auto")
    glacier_i = glacier_i.rio.reproject(f"EPSG:{get('epsg_utm_18s')}")
    glacier_list.append(glacier_i)

# Merge/Mosaic multiple rasters using merge_arrays method of rioxarray
merged_raster = merge_arrays(dataarrays = glacier_list, res = (res_hugonnet, res_hugonnet), crs=f"EPSG:{get('epsg_utm_18s')}", method='max')
merged_raster = merged_raster.where(merged_raster != nodata, np.nan)
merged_raster = merged_raster.rio.write_nodata(np.nan)

merged_raster = merged_raster.rio.reproject(f"EPSG:{get('epsg_wgs84')}")
merged_raster = merged_raster * 1000 / get('ice_to_water_density')  # from ice to water equivalent (mm)
merged_raster = merged_raster.fillna(0)
merged_raster.rio.to_raster(path_dhdt / outputs['glacier_dhdt'], compress="LZW")


### 2. **Millan et al. 2022**
- Processes glacier volume data from multiple tiles
- Reprojects to EPSG:32718 with bilinear resampling, then to EPSG:4326
- Merges tiles at 100m resolution
- Converts units from meters to km³
- **Output**: `volume_millan_2022_100m.tif`


In [ ]:
res_millan = get('res_millan')
list_files = glob.glob(str(path_thickness / inputs['glacier_volume_millan'] / "*.tif"))
glacier_list = []

# Read rasters file
for glacier in list_files:
    glacier_i = rio.open_rasterio(glacier)
    glacier_i = glacier_i.rio.reproject(f"EPSG:{get('epsg_utm_18s')}", resampling = Resampling.bilinear)
    glacier_list.append(glacier_i)

# Merge/Mosaic multiple rasters using merge_arrays method of rioxarray
merged_raster = merge_arrays(dataarrays = glacier_list, res = (res_millan, res_millan), crs=f"EPSG:{get('epsg_utm_18s')}", method='max')
merged_raster = merged_raster.where(merged_raster != nodata, np.nan)
merged_raster = merged_raster.rio.write_nodata(np.nan)
merged_raster = merged_raster * res_millan**2 / 1e9 # from m to km3
merged_raster = merged_raster.rio.reproject(f"EPSG:{get('epsg_wgs84')}")
merged_raster = merged_raster.fillna(0)
merged_raster.rio.to_raster(path_thickness / outputs['glacier_volume_millan'], compress="LZW")


### 3. **Farinotti et al. 2019**
- Processes ice thickness data for RGI regions 16 (Low Latitudes) and 17 (Southern Andes)
- Filters glaciers to study area (longitude: -80° to -50°, latitude: < 0°)
- Reprojects to EPSG:32718 at 200m resolution
- Processes each RGI region separately
- **Output**: `thickness_farinotti_2019_[region].tif`

In [ ]:
res_farinotti = get('res_farinotti')
list_files = glob.glob(str(path_thickness / inputs['glacier_thickness_farinotti'] / "*.tif"))

# subset glaciers
rgi_version = get('rgi_version')
RGI6_16 = gpd.read_file(path_outlines / f"{rgi_version}_16.shp")
RGI6_16 = RGI6_16[(RGI6_16['CenLon'] >= -80) & (RGI6_16['CenLon'] <= -50)] # remove glaciers outside the study area
RGI6_16 = RGI6_16[RGI6_16['CenLat'] < 0] 
RGI6_17 = gpd.read_file(path_outlines / f"{rgi_version}_17.shp")

for RGI in [RGI6_16, RGI6_17]:
    list_files_df = pd.DataFrame(list_files, columns=['dir'])
    list_files_df['RGIId'] = list_files_df['dir'].str.extract(r'(RGI60-\d{2}\.\d{5})')
    list_files_df = list_files_df[list_files_df['RGIId'].isin(RGI.RGIId)]

    glacier_list = []
    for glacier in list_files_df.dir:
        glacier_i = rio.open_rasterio(glacier)
        glacier_i = glacier_i.rio.reproject(f"EPSG:{get('epsg_utm_18s')}", resolution = (res_farinotti, res_farinotti), resampling = Resampling.bilinear)
        glacier_list.append(glacier_i)

    merged_raster = merge_arrays(dataarrays = glacier_list, res = (res_farinotti, res_farinotti), crs=f"EPSG:{get('epsg_utm_18s')}", method='max')
    merged_raster = merged_raster.where(merged_raster != nodata, np.nan)
    merged_raster = merged_raster.rio.write_nodata(np.nan)
    merged_raster.rio.to_raster(path_dhdt / outputs['glacier_thickness_farinotti'].format(region=RGI.O1Region[0]), compress="LZW")  

In [ ]:
# re-reads the per-region rasters written by the cell above, via the same config key
list_files = glob.glob(str(path_dhdt / outputs['glacier_thickness_farinotti'].format(region="*")))
glacier_list = []

# Read rasters file
for glacier in list_files:
    glacier_i = rio.open_rasterio(glacier)
    glacier_i = glacier_i.rio.reproject(f"EPSG:{get('epsg_utm_18s')}", resampling = Resampling.bilinear)
    glacier_list.append(glacier_i)

# Merge/Mosaic multiple rasters using merge_arrays method of rioxarray
merged_raster = merge_arrays(dataarrays = glacier_list, res = (res_farinotti, res_farinotti), crs=f"EPSG:{get('epsg_utm_18s')}", method='max')
merged_raster = merged_raster.where(merged_raster != nodata, np.nan)
merged_raster = merged_raster.rio.write_nodata(np.nan)
merged_raster = merged_raster * res_farinotti**2 / 1e9 # from m to km3
merged_raster = merged_raster.rio.reproject(f"EPSG:{get('epsg_wgs84')}")
merged_raster = merged_raster.fillna(0)
merged_raster.rio.to_raster(path_thickness / outputs['glacier_volume_farinotti'], compress="LZW")  